## tl;dr

- Apple M4에서 기존 2단계/3단계의 이미지 변환 커널은 문항당 각각 약 0.07초/0.15초였다.
- 동일한 실제 crop 3건에서 Lite 중앙값은 1.00초, Standard는 11.32초로 Lite가 약 11.3배 빨랐다.
- 정답 원본이 있는 합성 열화 세트에서 Upscayl Standard의 기술 충실도는 92.36점으로 2단계 76.62점보다 높았다. 3단계 점수는 페이지 장식 제거와 획 강화까지 변화로 계산되어 주관적 가독성 점수로 해석하면 안 된다.
- 권장안은 **3단계를 기본값으로 유지하고, 저해상도 crop에만 Upscayl Lite를 선택 적용**하는 것이다.

## Context & Methods

이 노트북은 EDB의 2단계, 3단계, 로컬 Upscayl을 품질·시간·비용 관점에서 비교한다. 대상 장치는 Apple M4 10-core GPU이며 측정일은 2026-07-14이다.

### Key Assumptions

- 최종 비교 폭은 1600px이다.
- 기술 충실도 점수는 허용 오차 edge F1 40%, 잉크 IoU 35%, alpha 유사도 25%의 합성 지표다.
- 합성 8건만 고해상도 정답 원본을 갖는다.
- 실제 국어·수학·과학 crop 6건은 원본 대비 구조 변화와 시간만 평가한다.
- 클라우드 API 호출은 하지 않았으며 가격은 공식 가격표의 출력 비용을 사용한다.
- 원화 환산은 계획 환율 1 USD = 1,400 KRW를 사용한다.

In [1]:
from pathlib import Path
import csv, json, statistics
from collections import defaultdict
from IPython.display import Markdown, display

ROOT = Path.cwd()
DATA = ROOT / "docs" / "upscayl_benchmark"

def load_csv(name):
    with (DATA / name).open(encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def md_table(rows, fields):
    header = "| " + " | ".join(fields) + " |"
    divider = "|" + "|".join(["---"] * len(fields)) + "|"
    body = ["| " + " | ".join(str(row.get(field, "")) for field in fields) + " |" for row in rows]
    return "\n".join([header, divider, *body])

synthetic = load_csv("synthetic_results.csv")
real = load_csv("real_results.csv")
models = load_csv("model_pilot_results.csv")
summary = json.loads((DATA / "summary.json").read_text(encoding="utf-8"))

print({
    "synthetic_rows": len(synthetic),
    "real_rows": len(real),
    "model_rows": len(models),
    "device": summary["device"],
})

{'synthetic_rows': 24, 'real_rows': 18, 'model_rows': 9, 'device': 'Apple M4 (10-core GPU), macOS arm64'}


## Data

In [2]:
display(Markdown("### 합성 정답 세트 요약\n\n" + md_table(
    summary["synthetic_summary"],
    ["method", "samples", "median_seconds", "mean_technical_fidelity_score", "mean_edge_f1", "mean_ink_iou"],
)))

display(Markdown("### 실제 crop 구조 보존 요약\n\n" + md_table(
    summary["real_summary"],
    ["method", "samples", "median_seconds", "mean_technical_fidelity_score", "mean_edge_f1", "mean_ink_iou"],
)))

display(Markdown("### Upscayl 모델 파일럿\n\n" + md_table(
    summary["model_summary"],
    ["method", "samples", "median_seconds", "mean_technical_fidelity_score", "mean_edge_f1", "mean_ink_iou"],
)))

### 합성 정답 세트 요약

| method | samples | median_seconds | mean_technical_fidelity_score | mean_edge_f1 | mean_ink_iou |
|---|---|---|---|---|---|
| stage2 | 8 | 0.032 | 76.62 | 0.7605 | 0.6251 |
| stage3 | 8 | 0.0796 | 64.06 | 0.6162 | 0.4426 |
| upscayl-standard-4x | 8 | 5.029 | 92.36 | 0.993 | 0.798 |

### 실제 crop 구조 보존 요약

| method | samples | median_seconds | mean_technical_fidelity_score | mean_edge_f1 | mean_ink_iou |
|---|---|---|---|---|---|
| stage2 | 6 | 0.0673 | 93.63 | 0.9999 | 0.8291 |
| stage3 | 6 | 0.1494 | 72.33 | 0.7962 | 0.4785 |
| upscayl-standard-4x | 6 | 11.0899 | 89.8 | 0.9982 | 0.7262 |

### Upscayl 모델 파일럿

| method | samples | median_seconds | mean_technical_fidelity_score | mean_edge_f1 | mean_ink_iou |
|---|---|---|---|---|---|
| ultrasharp-4x | 3 | 11.384 | 91.27 | 0.9974 | 0.7672 |
| upscayl-lite-4x | 3 | 1.0016 | 90.99 | 0.9973 | 0.7595 |
| upscayl-standard-4x | 3 | 11.3217 | 89.98 | 0.9973 | 0.7319 |

## Results

In [3]:
def group_mean(rows, field):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["method"]].append(float(row[field]))
    return {method: statistics.mean(values) for method, values in grouped.items()}

sharpness_synthetic = group_mean(synthetic, "sharpness")
sharpness_real = group_mean(real, "sharpness")
display(Markdown(md_table([
    {
        "method": method,
        "synthetic_sharpness": round(sharpness_synthetic[method], 1),
        "real_sharpness": round(sharpness_real[method], 1),
    }
    for method in sorted(sharpness_real)
], ["method", "synthetic_sharpness", "real_sharpness"])))

| method | synthetic_sharpness | real_sharpness |
|---|---|---|
| stage2 | 323.3 | 752.5 |
| stage3 | 686.4 | 1948.4 |
| upscayl-standard-4x | 1170.4 | 1771.4 |

In [4]:
# Local sequential throughput uses measured median kernel/runtime latency.
latency_seconds = {
    "2단계": 0.0673,
    "3단계": 0.1494,
    "Upscayl Lite": 1.0016,
    "Upscayl Standard": 11.0899,
    "Upscayl Ultrasharp": 11.3840,
}

volumes = [100, 1000, 10000]
throughput_rows = []
for method, seconds in latency_seconds.items():
    for volume in volumes:
        throughput_rows.append({
            "method": method,
            "images": volume,
            "sequential_minutes": round(seconds * volume / 60, 2),
            "sequential_hours": round(seconds * volume / 3600, 2),
        })

display(Markdown(md_table(
    throughput_rows,
    ["method", "images", "sequential_minutes", "sequential_hours"],
)))

| method | images | sequential_minutes | sequential_hours |
|---|---|---|---|
| 2단계 | 100 | 0.11 | 0.0 |
| 2단계 | 1000 | 1.12 | 0.02 |
| 2단계 | 10000 | 11.22 | 0.19 |
| 3단계 | 100 | 0.25 | 0.0 |
| 3단계 | 1000 | 2.49 | 0.04 |
| 3단계 | 10000 | 24.9 | 0.41 |
| Upscayl Lite | 100 | 1.67 | 0.03 |
| Upscayl Lite | 1000 | 16.69 | 0.28 |
| Upscayl Lite | 10000 | 166.93 | 2.78 |
| Upscayl Standard | 100 | 18.48 | 0.31 |
| Upscayl Standard | 1000 | 184.83 | 3.08 |
| Upscayl Standard | 10000 | 1848.32 | 30.81 |
| Upscayl Ultrasharp | 100 | 18.97 | 0.32 |
| Upscayl Ultrasharp | 1000 | 189.73 | 3.16 |
| Upscayl Ultrasharp | 10000 | 1897.33 | 31.62 |

In [5]:
USD_KRW = 1400
api_prices = {
    "Gemini 3.1 Flash Image 1K": 0.067,
    "Gemini 3.1 Flash Image 2K": 0.101,
    "GPT Image 2 high landscape": 0.165,
    "GPT Image 2 high square": 0.211,
}
cost_rows = []
for method, usd_per_image in api_prices.items():
    for volume in volumes:
        usd = usd_per_image * volume
        cost_rows.append({
            "method": method,
            "images": volume,
            "usd_output_cost": round(usd, 2),
            "krw_output_cost": round(usd * USD_KRW),
        })

display(Markdown(md_table(
    cost_rows,
    ["method", "images", "usd_output_cost", "krw_output_cost"],
)))

| method | images | usd_output_cost | krw_output_cost |
|---|---|---|---|
| Gemini 3.1 Flash Image 1K | 100 | 6.7 | 9380 |
| Gemini 3.1 Flash Image 1K | 1000 | 67.0 | 93800 |
| Gemini 3.1 Flash Image 1K | 10000 | 670.0 | 938000 |
| Gemini 3.1 Flash Image 2K | 100 | 10.1 | 14140 |
| Gemini 3.1 Flash Image 2K | 1000 | 101.0 | 141400 |
| Gemini 3.1 Flash Image 2K | 10000 | 1010.0 | 1414000 |
| GPT Image 2 high landscape | 100 | 16.5 | 23100 |
| GPT Image 2 high landscape | 1000 | 165.0 | 231000 |
| GPT Image 2 high landscape | 10000 | 1650.0 | 2310000 |
| GPT Image 2 high square | 100 | 21.1 | 29540 |
| GPT Image 2 high square | 1000 | 211.0 | 295400 |
| GPT Image 2 high square | 10000 | 2110.0 | 2954000 |

In [6]:
# Electricity-only estimate. Hardware depreciation and labor are excluded.
POWER_WATTS = 30
ELECTRICITY_KRW_PER_KWH = 200
local_cost_rows = []
for method, seconds in latency_seconds.items():
    if not method.startswith("Upscayl"):
        continue
    for volume in volumes:
        kwh = POWER_WATTS * seconds * volume / 3_600_000
        local_cost_rows.append({
            "method": method,
            "images": volume,
            "estimated_kwh": round(kwh, 5),
            "electricity_krw": round(kwh * ELECTRICITY_KRW_PER_KWH, 2),
        })

display(Markdown(md_table(
    local_cost_rows,
    ["method", "images", "estimated_kwh", "electricity_krw"],
)))

| method | images | estimated_kwh | electricity_krw |
|---|---|---|---|
| Upscayl Lite | 100 | 0.00083 | 0.17 |
| Upscayl Lite | 1000 | 0.00835 | 1.67 |
| Upscayl Lite | 10000 | 0.08347 | 16.69 |
| Upscayl Standard | 100 | 0.00924 | 1.85 |
| Upscayl Standard | 1000 | 0.09242 | 18.48 |
| Upscayl Standard | 10000 | 0.92416 | 184.83 |
| Upscayl Ultrasharp | 100 | 0.00949 | 1.9 |
| Upscayl Ultrasharp | 1000 | 0.09487 | 18.97 |
| Upscayl Ultrasharp | 10000 | 0.94867 | 189.73 |

## Takeaways

1. **2단계는 가장 안전하고 빠른 기본 변환이다.** 이미 선명한 원본은 추가 초해상도 없이도 충분하다.
2. **3단계는 가독성과 페이지 장식 제거를 위한 제품 처리다.** 기술 충실도 점수가 낮은 것은 실패라기보다 의도적인 crop·획 변화가 포함되기 때문이다.
3. **Upscayl은 저해상도 복원 효과가 확실하지만 Standard는 운영 시간이 크다.** Lite는 동일한 표본 3건의 중앙값 비교에서 Standard보다 약 11.3배 빨랐고 구조 보존도 동등 이상이었다.
4. **API 재구성은 예외 처리로 남기는 편이 경제적이다.** 1,000문항을 2K Gemini로 처리하면 출력만 약 $101이며, GPT Image 2 high는 약 $165~$211에 입력 비용이 추가된다.
5. **권장 라우팅은 3단계 기본 + 조건부 Upscayl Lite + API 최후 fallback이다.**